In [ ]:
%matplotlib inline
# changed from %matplotlib notebook as graphs werent being plotted 

In [ ]:
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Seq import Seq 
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import numpy as np
from numpy import random as rd
from matplotlib import pyplot as plt

In [ ]:
from src.estimalign import estimalign
from src.logit_link import logit_partial_scores
from src.optimization import create_powerstep, create_constant_step

# Data

In [ ]:
from miRBench.dataset import list_datasets, get_dataset_df

In [ ]:
hejret_train = get_dataset_df(list_datasets()[0], split="train")
hejret_test = get_dataset_df(list_datasets()[0], split="test")

In [ ]:
mirlist = hejret_train['noncodingRNA']
mirlist = [Seq(seq) for seq in mirlist]
genelist = hejret_train['gene']
genelist = [Seq(seq).reverse_complement() for seq in genelist]

# Optimization

### Simple model on miRNA alignments:

In [ ]:
true_match = 1
true_mismatch = -1
true_gapopen = -1.2
true_gapext = -0.1

In [ ]:
aligner = PairwiseAligner()
aligner.mode = 'local'
aligner.open_gap_score = true_gapopen
aligner.extend_gap_score = true_gapext
aligner.match = true_match
aligner.mismatch = true_mismatch
# aligner.end_gap_score = 0

In [ ]:
scores = np.array([aligner.score(a, b) for a, b in zip(mirlist, genelist)])

In [ ]:
plt.figure()
plt.hist(scores, bins=100)
plt.show()

In [ ]:
true_alpha = -9
## true_alpha = -np.median(scores)

In [ ]:
logit_scores = logit_partial_scores(scores, true_alpha)

In [ ]:
plt.figure()
plt.hist(logit_scores, bins=100)
plt.show()

### Test Run

In [ ]:
labels = rd.rand(len(mirlist))
labels = labels <= logit_scores
labels

In [ ]:
true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))
print('Sum of log-logit scores:', np.sum(np.log(logit_scores)))
print('True LogL:', true_logL) ## This is different 

In [ ]:
plt.figure()
plt.plot(scores, labels, 'r.', alpha=0.005)

In [ ]:
plt.figure()
plt.plot(logit_scores, labels, 'r.', alpha=0.005)

In [ ]:
const_step = create_constant_step(0.00001)
# powerstep = create_powerstep(0.00001, power=0.5, burnin=0)
# powerstep = create_powerstep(0.00001, power=-0.5, burnin=0)

In [ ]:
NITER = 50 # original 50

In [ ]:
params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='simple',
                    verbose=True, max_iter=NITER,
                    stochastic_factor=None,
                    num_threads = 16)

In [ ]:
print(params['final_loglik'])

In [ ]:
print(params['final_loglik'])

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(np.arange(NITER), params['subgradient_l2_trajectory'])
plt.plot([0, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')


plt.subplot(222)
plt.plot(np.arange(NITER+1), params['loglik_trajectory'])
plt.plot([0, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.subplot(223)
plt.plot(np.arange(NITER//2, NITER), params['subgradient_l2_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')

plt.subplot(224)
plt.plot(np.arange(NITER//2, NITER+1), params['loglik_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.tight_layout()

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(np.arange(NITER), params['subgradient_l2_trajectory'])
plt.plot([0, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')


plt.subplot(222)
plt.plot(np.arange(NITER+1), params['loglik_trajectory'])
plt.plot([0, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.subplot(223)
plt.plot(np.arange(NITER//2, NITER), params['subgradient_l2_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')

plt.subplot(224)
plt.plot(np.arange(NITER//2, NITER+1), params['loglik_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.tight_layout()

In [ ]:
print(true_match, params['match_score'])
print(true_mismatch, params['mismatch_score'])
print(true_gapopen, params['open_gap_score'])
print(true_gapext, params['extend_gap_score'])
print(true_alpha, params['alpha'])

### Additional plotting

In [ ]:
# ## Freezing dataset values so true loglikliehood is not recomputed after every iteration 

# rng = np.random.default_rng(0)

# # fixed labels 
# labels_frozen = rng.random(len(mirlist)) <= logit_partial_scores(scores, true_alpha)

# # fixed model probabilities for true log-likelihood
# logit_scores_frozen = logit_partial_scores(scores, true_alpha)

# # compute true log-likelihood ONCE
# true_logL_frozen = (
#     np.sum(np.log(logit_scores_frozen[labels_frozen])) +
#     np.sum(np.log(1 - logit_scores_frozen[~labels_frozen]))
# )

In [ ]:
# # Run estimalign across different step sizes and plot:
# # (1) logL trajectories to see convergence behavior
# # (2) scatter of convergence speed vs final parameter error to find the best step size

# step_sizes = [0.000005, 0.00001, 0.00005, 0.0001, 0.0005]
# task1_results = {}

# for ss in step_sizes:
#     step = create_constant_step(ss)
#     p = estimalign(mirlist, genelist, labels_frozen, ## labels_frozen now used 
#                    stepfunction=step, aligner_mode='local',
#                    substitution_mode='simple', verbose=False,
#                    max_iter=NITER, stochastic_factor=None, num_threads=16)
#     task1_results[ss] = p

# # Trajectory plot
# plt.figure()
# for ss, p in task1_results.items():
#     plt.plot(p['loglik_trajectory'], label=f'step={ss}')
# plt.axhline(true_logL, color='k', linestyle='--', label='True LogL')
# plt.xlabel('Iteration'); plt.ylabel('LogL')
# plt.title('LogL Trajectories by Step Size')
# plt.legend(); plt.tight_layout(); plt.show()

# # Scatter: convergence speed vs final parameter error
# def param_error(p):
#     return (abs(p['match_score'] - true_match) +
#             abs(p['mismatch_score'] - true_mismatch) +
#             abs(p['open_gap_score'] - true_gapopen) +
#             abs(p['extend_gap_score'] - true_gapext) +
#             abs(p['alpha'] - true_alpha))

# # convergence = first iteration where subgradient L2 norm drops below threshold
# threshold = 0.01
# conv_iters = []
# errors = []
# for ss, p in task1_results.items():
#     grads = p['subgradient_l2_trajectory']
#     conv = next((i for i, g in enumerate(grads) if g < threshold), NITER)
#     conv_iters.append(conv)
#     errors.append(param_error(p))

# plt.figure()
# for i, ss in enumerate(step_sizes):
#     plt.scatter(conv_iters[i], errors[i], label=f'step={ss}', s=100)
# plt.xlabel('Iterations to Converge')
# plt.ylabel('Total Parameter Error')
# plt.title('Convergence Speed vs Parameter Error')
# plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# # Run estimalign 20 times with different random seeds and plot:
# # 1) a histogram of mean parameter error per run
# # 2) per-parameter boxplots to show which parameters are reliably recovered

# N_SEEDS = 20
# best_step = 0.000005 # best step size from task 1 

# # If we treat alpha as a single constant parameter then there wont be any error, so I removed it
# # For some reason doing that changes all of the error to the same bins  

# param_names = ['match_score', 'mismatch_score', 'open_gap_score', 'extend_gap_score']
# true_values = [true_match, true_mismatch, true_gapopen, true_gapext]

# per_param_errors = {name: [] for name in param_names}
# mean_errors = []

# for seed in range(N_SEEDS):
#     rd.seed(seed)

#     p = estimalign(mirlist, genelist, labels_frozen, ## changed labels to labels_frozen
#                    stepfunction=create_constant_step(best_step), aligner_mode='local',
#                    substitution_mode='simple', verbose=False,
#                    max_iter=NITER, stochastic_factor=None, num_threads=16)

#     errors = []
#     for name, true_val in zip(param_names, true_values):
#         err = abs(p[name] - true_val)
#         per_param_errors[name].append(err)
#         errors.append(err)

#     mean_errors.append(np.mean(errors))

# # Histogram: mean error per run
# plt.figure()
# plt.hist(mean_errors, bins=10, edgecolor='black')
# plt.xlabel('Mean Absolute Parameter Error')
# plt.ylabel('Count')
# plt.title('Distribution of Mean Parameter Error Across 20 Runs')
# plt.tight_layout()
# plt.show()

# # Boxplots: per-parameter error 
# plt.figure()
# plt.boxplot(per_param_errors.values(), labels=param_names, showfliers=True)
# plt.ylabel('Mean Absolute Error')
# plt.title('Per-Parameter Recovery Error Across 20 Runs')
# plt.xticks(rotation=15)
# plt.tight_layout()
# plt.show()

### Step function parameters experiment

In [ ]:
labels = rd.rand(len(mirlist)) <= logit_scores

In [ ]:
true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))

In [ ]:
steplengths = np.linspace(0.000005, 0.00005, num=10)
steplengths

In [ ]:
NITER = 200

In [ ]:
# Get the best step size from the first experiment (lowest pen wins)
def step_penalty(logL, true_logL):
    return abs(logL - true_logL)

In [ ]:
# Get the iteration at which convergence happens (speed)
# by also measuring speed, a tradeoff between speed and quality (measured above) happes
def get_convergence_iter(params, true_logL, tol=1.0):
    for i, logL in enumerate(params['loglik_trajectory']):
        if step_penalty(logL, true_logL) < tol:
            return i
    return len(params['loglik_trajectory']) - 1

In [ ]:
estimalign_results = []

conv_iters = []
penalties = []

for stepl in steplengths:
    const_step = create_constant_step(stepl)
    
    params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='simple',
                    gap_mode = 'affine',
                    verbose=False, max_iter=NITER,
                    stochastic_factor=0.001,
                    num_threads = 16)
    estimalign_results.append(params)

    final_logL = params['loglik_trajectory'][-1]

    penalty = step_penalty(final_logL, true_logL)
    penalties.append(penalty)

    conv_iter = get_convergence_iter(params, true_logL, tol = 0.5)
    conv_iters.append(conv_iter)

    if conv_iter < NITER:
        print(f"step={stepl:.2e}, penalty={penalty:.4f}, converged at iteration {conv_iter}")
    else:
        print(f"step={stepl:.2e}, penalty={penalty:.4f}, did NOT converge") 

best_step = steplengths[np.argmin(penalties)]
print(f"\nBest step to be used for replicate experiment: ", best_step)

In [ ]:
plt.figure()
for params in estimalign_results:
    plt.plot(np.arange(NITER+1), params['loglik_trajectory'], alpha=0.5)
plt.plot([0, NITER], [true_logL, true_logL], 'k--')
plt.legend([f"{sl:.1e}" for sl in steplengths])
plt.tight_layout()
# plt.savefig('path', dpi=160)

In [ ]:
plt.figure()
for params in estimalign_results:
    plt.plot(np.arange(NITER+1), params['loglik_trajectory'], alpha=0.5)
plt.plot([0, NITER], [true_logL, true_logL], 'k--')
plt.legend([f"{sl:.1e}" for sl in steplengths])
plt.xlim(0, 5)
plt.tight_layout()

### Replicates

In [ ]:
REPS = 20
NITER = 200

In [ ]:
const_step = create_constant_step(best_step) # updated to choose best step from prev exp
# const_step = create_constant_step(0.00001)
# powerstep = create_powerstep(0.00001, power=0.5, burnin=0)
# powerstep = create_powerstep(0.00001, power=-0.5, burnin=0)

In [ ]:
estimalign_results = []
true_logLs = []
conv_iters_rep = [] # Checks if the chosen step size is stable in convergence across different datasets

for _ in range(REPS):
    labels = rd.rand(len(mirlist)) <= logit_scores
    true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))
    true_logLs.append(true_logL)
    params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='simple',
                    gap_mode = 'affine',
                    verbose=False, max_iter=NITER,
                    stochastic_factor=0.001,
                    num_threads = 16)
    estimalign_results.append(params)

    conv_iter = get_convergence_iter(params, true_logL, tol=1.0)
    conv_iters_rep.append(conv_iter)

for i, it in enumerate(conv_iters_rep):
    if it < NITER - 1:
        print(f"rep {i} converged at iteration {it}")
    else:
        print(f"rep {i} did NOT converge")

In [ ]:
plt.figure(figsize=(7.5, 2.1))
plt.subplot(121)
for params in estimalign_results:
    plt.plot(np.arange(NITER+1), params['loglik_trajectory'], alpha=0.2, color='b')
plt.subplot(122)
plt.plot([0, NITER], [0, 0], 'k--')
for params, tlL in zip(estimalign_results, true_logLs):
    plt.plot(np.arange(NITER+1), tlL - params['loglik_trajectory'], alpha=0.2, color='b')

### General matrix, affine gap penalty

In [ ]:
true_gapopen = -1.2
true_gapext = -0.1
true_substitution = substitution_matrices.Array(alphabet='ACTG', 
                                          data=np.array([
                                              [1, -0.3, -1, -0.8], 
                                              [-0.6, 1.2, -0.3, -1], 
                                              [-1.2, -0.4, 1, -0.8], 
                                              [-0.4, -1.4, -0.9, 1.3]]))

In [ ]:
aligner = PairwiseAligner()
aligner.mode = 'local'
aligner.open_gap_score = true_gapopen
aligner.extend_gap_score = true_gapext
aligner.substitution_matrix = true_substitution

In [ ]:
scores = np.array([aligner.score(a, b) for a, b in zip(mirlist, genelist)])

In [ ]:
plt.figure()
plt.hist(scores, bins=100)
plt.show()

In [ ]:
true_alpha = -12
logit_scores = logit_partial_scores(scores, true_alpha)

In [ ]:
plt.figure()
plt.hist(logit_scores, bins=100)
plt.show()

In [ ]:
labels = rd.rand(len(mirlist))
labels = labels <= logit_scores
true_logL = np.sum(np.log(logit_scores[labels]))+np.sum(np.log(1-logit_scores[~labels]))
print('Sum of log-logit scores:', np.sum(np.log(logit_scores)))
print('True LogL:', true_logL)

In [ ]:
const_step = create_constant_step(0.00005)
# powerstep = create_powerstep(0.00005, power=0.5, burnin=0)
powerstep = create_powerstep(0.00002, power=-0.1, burnin=0)

In [ ]:
NITER = 200

In [ ]:
params = estimalign(mirlist, genelist, labels, 
                    stepfunction=const_step,
                    aligner_mode='local',
                    substitution_mode='general',
                    gap_mode='affine', 
                    stochastic_factor=0.01,
                    verbose=True, max_iter=NITER,
                    num_threads = 24)

In [ ]:
print(params['final_loglik'])

In [ ]:
plt.figure()
plt.subplot(221)
plt.plot(np.arange(NITER), params['subgradient_l2_trajectory'])
plt.plot([0, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')


plt.subplot(222)
plt.plot(np.arange(NITER+1), params['loglik_trajectory'])
plt.plot([0, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')


plt.subplot(223)
plt.plot(np.arange(NITER//2, NITER), params['subgradient_l2_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER], [0, 0], 'k--')
plt.title('Subgradient L2 norm trajectory')

plt.subplot(224)
plt.plot(np.arange(NITER//2, NITER+1), params['loglik_trajectory'][NITER//2:])
plt.plot([NITER//2, NITER+1], [true_logL, true_logL], 'k--')
plt.title('LogLikelihood trajectory')

plt.tight_layout()

In [ ]:
print(true_gapopen, params['open_gap_score'])
print(true_gapext, params['extend_gap_score'])
print(true_alpha, params['alpha'])
true_subs_vector = []
param_subs_vector = []
for char1 in true_substitution.alphabet:
    for char2 in true_substitution.alphabet:
        true_v = true_substitution[char1, char2]
        true_subs_vector.append(true_v)
        estim_v = params['substitution_matrix'][char1, char2]
        param_subs_vector.append(estim_v)
        print(char1, char2, true_v, estim_v)
        
print(np.corrcoef(true_subs_vector, param_subs_vector))
print(np.mean(np.abs(np.array(true_subs_vector)- np.array(param_subs_vector))))

In [ ]:
plt.figure()
plt.plot(true_subs_vector, param_subs_vector, 'r.')

In [ ]:
from src.optimization import get_initial_estimate

In [ ]:
get_initial_estimate(params['alignments'], labels, substitution_mode='general', gap_mode='affine',
                     alphabet=true_substitution.alphabet)

In [ ]:
true_substitution

In [ ]:
estim_substitution = true_substitution.copy()
for char1 in true_substitution.alphabet:
    for char2 in true_substitution.alphabet:
        estim_substitution[char1, char2] = params['substitution_matrix'][char1, char2]

In [ ]:
estim_substitution